In [7]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

In [8]:
df = pd.read_csv("HMS_2024-2025_PUBLIC_instchars .csv", low_memory = False)

In [12]:
column_list = df.columns.tolist()
print(column_list)

['StartDate', 'Finished', 'RecordedDate', 'responseid', 'schoolnum', 'age', 'gender_male', 'gender_female', 'gender_queer', 'gender_nonbin', 'gender_trans', 'gender_prefnoresp', 'gender_selfID', 'gender_text', 'sex_birth', 'sexual_h', 'sexual_l', 'sexual_g', 'sexual_bi', 'sexual_queer', 'sexual_quest', 'sexual_asexual', 'sexual_pan', 'sexual_prefnoresp', 'sexual_selfID', 'sexual_text', 'race_black', 'race_ainaan', 'race_asian', 'race_his', 'race_pi', 'race_mides', 'race_white', 'race_other', 'race_other_text', 'black_african', 'black_africanam', 'black_caribbean', 'black_afrolatin', 'black_other', 'asian_east', 'asian_southeast', 'asian_south', 'asian_filipin', 'asian_other', 'his_mexican', 'his_centralam', 'his_southam', 'his_caribbean', 'his_spainport', 'his_other', 'international', 'age_US', 'state_9thgrade', 'state_10thgrade', 'state_11thgrade', 'state_12thgrade', 'year_9thgrade', 'year_10thgrade', 'year_11thgrade', 'year_12thgrade', 'sexed_TGE', 'sexed_LGBQ', 'housing_worry', 'dep

In [5]:
# PHQ-9 and GAD-7 totals
phq_items = [f"phq9_{i}" for i in range(1, 10)]
gad_items = [f"gad7_{i}" for i in range(1, 8)]

df["phq9_total"] = df[phq_items].sum(axis=1)
df["gad7_total"] = df[gad_items].sum(axis=1)

# Binary outcomes (moderate/severe)
df["dep_any"] = (df["phq9_total"] >= 10).astype(int)
df["anx_any"] = (df["gad7_total"] >= 10).astype(int)


In [8]:
fin_vars = [
    "FinStress", "fincur", "finpast",
    "food_worry", "food_notlast",
    "afford_house",
    "pay_worry1", "pay_worry2", "pay_worry3"
]

df["fin_stress_index"] = df[fin_vars].mean(axis=1)


In [11]:
hous_vars = [
    "housing_worry", "hous_conc",
    "housing_inst_5", "housing_inst_6", "housing_inst_7",
    "housing_inst_8", "housing_inst_9", "housing_inst_10", "housing_inst_11"
]

df["housing_index"] = df[hous_vars].mean(axis=1)


In [12]:
acad_stress_vars = ["aca_impa", "stress1", "stress2", "stress3", "stress4"]
df["academic_stress_index"] = df[acad_stress_vars].mean(axis=1)


In [14]:
# List of items where higher = more positive outlook
pos_outlook_vars = [
    "persist",          # more likely to stay/finish
    "Retention",        # if coded so higher = more likely to return
    "ret_confid_y",     # confidence in returning
    "compet_cl", "compet_sch", "compet_field",  # feel competent
    "achieve1", "achieve2", "achieve3", "achieve4"  # achievement beliefs
]

# Doubt item where higher = more doubt (we reverse it)
# Adjust if your coding is different
df["doubt_school_1_rev"] = df["doubt_school_1"].max() - df["doubt_school_1"]

all_outlook_vars = pos_outlook_vars + ["doubt_school_1_rev"]

# Convert all columns to numeric before calculating mean
# This will handle any string values by converting them to NaN
for col in all_outlook_vars:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Now calculate the mean, ignoring NaN values
df["academic_future_outlook_index"] = df[all_outlook_vars].mean(axis=1)

# Standardize for easier interpretation
df["academic_future_outlook_std"] = (
    df["academic_future_outlook_index"] - df["academic_future_outlook_index"].mean()
) / df["academic_future_outlook_index"].std()


In [15]:
def race_cat(row):
    if row.get("race_black", 0) == 1:
        return "Black"
    if row.get("race_his", 0) == 1:
        return "Hispanic"
    if row.get("race_asian", 0) == 1:
        return "Asian"
    if row.get("race_white", 0) == 1:
        return "White"
    return "Other"

df["race_cat"] = df.apply(race_cat, axis=1)

def gender_cat(row):
    if row.get("gender_male", 0) == 1:
        return "Man"
    if row.get("gender_female", 0) == 1:
        return "Woman"
    if row.get("gender_nonbin", 0) == 1 or row.get("gender_queer", 0) == 1 or row.get("gender_trans", 0) == 1:
        return "TGNC"
    return "Other"

df["gender_cat"] = df.apply(gender_cat, axis=1)

df["race_cat"] = df["race_cat"].astype("category")
df["gender_cat"] = df["gender_cat"].astype("category")


In [16]:
desc_vars = [
    "phq9_total", "gad7_total",
    "fin_stress_index", "housing_index",
    "academic_stress_index", "academic_future_outlook_index"
]
print(df[desc_vars].describe())

corr = df[desc_vars].corr()
print(corr)


         phq9_total    gad7_total  fin_stress_index  housing_index  \
count  84735.000000  84735.000000      79456.000000   80631.000000   
mean      15.325238     12.730029          2.413163       1.188852   
std        7.866245      7.096944          0.619779       0.483980   
min        0.000000      0.000000          1.000000       0.000000   
25%       11.000000      8.000000          2.000000       1.000000   
50%       15.000000     12.000000          2.333333       1.000000   
75%       20.000000     18.000000          3.000000       1.000000   
max       36.000000     28.000000          5.000000       3.000000   

       academic_stress_index  academic_future_outlook_index  
count           75061.000000                   75957.000000  
mean                2.259021                       1.416097  
std                 1.036276                       0.608450  
min                 1.000000                       0.666667  
25%                 1.000000                       1.000000

In [17]:
def pseudo_r2(model):
    return 1 - model.llf / model.llnull

In [18]:
model1_dep = smf.logit(
    "dep_any ~ academic_stress_index + academic_future_outlook_std + C(race_cat) + C(gender_cat)",
    data=df
).fit()
print(model1_dep.summary())
print("Depression Model 1 pseudo-R2:", pseudo_r2(model1_dep))


Optimization terminated successfully.
         Current function value: 0.239874
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:                dep_any   No. Observations:                72569
Model:                          Logit   Df Residuals:                    72559
Method:                           MLE   Df Model:                            9
Date:                Tue, 27 Jan 2026   Pseudo R-squ.:                  0.1522
Time:                        15:54:13   Log-Likelihood:                -17407.
converged:                       True   LL-Null:                       -20533.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept                      -0.2485      0.052     -4.761      0.000   

In [19]:
model2_dep = smf.logit(
    "dep_any ~ academic_stress_index + academic_future_outlook_std + "
    "fin_stress_index + housing_index + C(race_cat) + C(gender_cat)",
    data=df
).fit()
print(model2_dep.summary())
print("Depression Model 2 pseudo-R2:", pseudo_r2(model2_dep))
print("Δ pseudo-R2:", pseudo_r2(model2_dep) - pseudo_r2(model1_dep))


Optimization terminated successfully.
         Current function value: 0.235871
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:                dep_any   No. Observations:                71772
Model:                          Logit   Df Residuals:                    71760
Method:                           MLE   Df Model:                           11
Date:                Tue, 27 Jan 2026   Pseudo R-squ.:                  0.1628
Time:                        15:54:17   Log-Likelihood:                -16929.
converged:                       True   LL-Null:                       -20220.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept                       1.0290      0.106      9.678      0.000   

In [20]:
model1_anx = smf.logit(
    "anx_any ~ academic_stress_index + academic_future_outlook_std + C(race_cat) + C(gender_cat)",
    data=df
).fit()
print(model1_anx.summary())
print("Anxiety Model 1 pseudo-R2:", pseudo_r2(model1_anx))

model2_anx = smf.logit(
    "anx_any ~ academic_stress_index + academic_future_outlook_std + "
    "fin_stress_index + housing_index + C(race_cat) + C(gender_cat)",
    data=df
).fit()
print(model2_anx.summary())
print("Anxiety Model 2 pseudo-R2:", pseudo_r2(model2_anx))
print("Δ pseudo-R2:", pseudo_r2(model2_anx) - pseudo_r2(model1_anx))


Optimization terminated successfully.
         Current function value: 0.444975
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                anx_any   No. Observations:                72569
Model:                          Logit   Df Residuals:                    72559
Method:                           MLE   Df Model:                            9
Date:                Tue, 27 Jan 2026   Pseudo R-squ.:                  0.1889
Time:                        15:54:25   Log-Likelihood:                -32291.
converged:                       True   LL-Null:                       -39812.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept                      -1.8945      0.036    -52.176      0.000   

In [21]:
model_dep_race_int = smf.logit(
    "dep_any ~ academic_stress_index + academic_future_outlook_std + "
    "fin_stress_index * C(race_cat) + housing_index * C(race_cat) + "
    "C(gender_cat)",
    data=df
).fit()
print(model_dep_race_int.summary())


Optimization terminated successfully.
         Current function value: 0.235762
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:                dep_any   No. Observations:                71772
Model:                          Logit   Df Residuals:                    71752
Method:                           MLE   Df Model:                           19
Date:                Tue, 27 Jan 2026   Pseudo R-squ.:                  0.1632
Time:                        15:54:36   Log-Likelihood:                -16921.
converged:                       True   LL-Null:                       -20220.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                               coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------
Intercept                                    0.8

In [22]:
model_dep_gender_int = smf.logit(
    "dep_any ~ academic_stress_index + academic_future_outlook_std + "
    "fin_stress_index * C(gender_cat) + housing_index * C(gender_cat) + "
    "C(race_cat)",
    data=df
).fit()
print(model_dep_gender_int.summary())


Optimization terminated successfully.
         Current function value: 0.235789
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:                dep_any   No. Observations:                71772
Model:                          Logit   Df Residuals:                    71754
Method:                           MLE   Df Model:                           17
Date:                Tue, 27 Jan 2026   Pseudo R-squ.:                  0.1631
Time:                        15:54:44   Log-Likelihood:                -16923.
converged:                       True   LL-Null:                       -20220.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                              coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------
Intercept                                   0.8944